# Question 1: datetime Fundamentals and Time Series Indexing

This question focuses on datetime handling and time series indexing using patient vital signs data.

## Setup

In [7]:
import sys
!{sys.executable} -m pip install seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('default')
sns.set_style('whitegrid')

# Create output directory
os.makedirs('output', exist_ok=True)

## Part 1.1: Load and Explore Data

**Note:** This dataset contains realistic healthcare data characteristics:
- **200 patients** with daily vital signs over 1 year
- **Missing visits**: Patients miss approximately 5% of scheduled visits (realistic!)
- **Different start dates**: Not all patients start monitoring on January 1st (some join later)
- When selecting data by date ranges, you may find that some patients don't have data for certain periods - this is expected and realistic

In [9]:
# Load patient vital signs data
patient_vitals = pd.read_csv('data/patient_vitals.csv')

print("Patient vitals shape:", patient_vitals.shape)
print("Patient vitals columns:", patient_vitals.columns.tolist())

# Display sample data
print("\nPatient vitals sample:")
print(patient_vitals.head())
print("\nData summary:")
print(patient_vitals.describe())

# Check date range and missing data patterns
print(f"\nDate range: {patient_vitals['date'].min()} to {patient_vitals['date'].max()}")
print(f"Unique patients: {patient_vitals['patient_id'].nunique()}")
print(f"Total records: {len(patient_vitals)}")
print(f"Expected records (200 patients × 365 days): {200 * 365:,}")
print(f"Missing visits: ~{200 * 365 - len(patient_vitals):,} records")

Patient vitals shape: (18250, 7)
Patient vitals columns: ['date', 'patient_id', 'temperature', 'heart_rate', 'blood_pressure_systolic', 'blood_pressure_diastolic', 'weight']

Patient vitals sample:
         date patient_id  temperature  heart_rate  blood_pressure_systolic  \
0  2023-01-01      P0001    98.389672          71                      119   
1  2023-01-02      P0001    98.492046          67                      117   
2  2023-01-03      P0001    98.790163          70                      113   
3  2023-01-04      P0001    98.635781          74                      117   
4  2023-01-05      P0001    98.051660          67                      118   

   blood_pressure_diastolic     weight  
0                        84  68.996865  
1                        82  67.720215  
2                        78  67.846825  
3                        82  67.693993  
4                        83  68.228852  

Data summary:
        temperature    heart_rate  blood_pressure_systolic  \
count  182

## Part 1.2: datetime Operations

**TODO: Perform datetime operations**

In [10]:
import pandas as pd
import numpy as np
import os

# Make sure output folder exists
os.makedirs('output', exist_ok=True)

# Load data
patient_vitals = pd.read_csv('data/patient_vitals.csv')

# TODO: Convert date column to datetime
patient_vitals['date'] = pd.to_datetime(patient_vitals['date'])

# TODO: Set datetime column as index
patient_vitals = patient_vitals.set_index('date')

# TODO: Extract year, month, day components from datetime index
patient_vitals['year'] = patient_vitals.index.year 
patient_vitals['month'] = patient_vitals.index.month
patient_vitals['day'] = patient_vitals.index.day

# TODO: Calculate time differences (e.g., days since first measurement)
# Note: Since patients start at different times, calculate days_since_start per patient
# Hint: To use groupby on the 'date' column, temporarily reset the index, then set it back
patient_vitals_reset = patient_vitals.reset_index()
# Calculate days since each patient's first date
patient_vitals_reset['days_since_start'] = patient_vitals_reset.groupby('patient_id')['date'].transform(lambda x: (x - x.min()).dt.days)
# Set the date back as index
patient_vitals = patient_vitals_reset.set_index('date')
#          Use groupby('patient_id')['date'].transform(lambda x: (x - x.min()).dt.days)
#          Or use groupby('patient_id').apply() to calculate days from each patient's first date
#          Then: patient_vitals = patient_vitals_reset.set_index('date')
# patient_vitals['days_since_start'] = None  # Calculate from each patient's start date

# TODO: Create business day ranges for clinic visit schedules
clinic_dates = pd.date_range(start=patient_vitals.index.min(), end=patient_vitals.index.max())

# TODO: Create date ranges with different frequencies
daily_range = pd.date_range(start=patient_vitals.index.min(), end=patient_vitals.index.max(), freq='D')  # Daily schedule
weekly_range = pd.date_range(start=patient_vitals.index.min(), end=patient_vitals.index.max(), freq='W-MON')  # Weekly labs on Mondays
monthly_range = pd.date_range(start=patient_vitals.index.min(), end=patient_vitals.index.max(), freq='MS')  # Monthly checkups on the first day of each month

# TODO: Use date ranges to analyze visit patterns
# Check how many patient visits occurred on clinic business days vs weekends
patient_dates_set = set(patient_vitals.index.date)
clinic_dates_set = set(clinic_dates.date)
visits_on_clinic_days = len(patient_dates_set & clinic_dates_set)
visits_on_weekends = len(patient_dates_set) - visits_on_clinic_days
print(f"Visits on clinic business days: {visits_on_clinic_days}")
print(f"Visits on weekends: {visits_on_weekends}")
print(f"Total unique visit dates: {len(patient_dates_set)}")

# TODO: Save results as 'output/q1_datetime_analysis.csv'
# Create a DataFrame with datetime analysis results including:
# - date (datetime index or column)
# - year, month, day (extracted from datetime)
# - days_since_start (calculated time differences)
# - patient_id
# - At least one original column (e.g., temperature, heart_rate)
cols_to_save = ['patient_id', 'year', 'month', 'day', 'days_since_start']
# Add one original vital sign column if exists
for col in ['temperature', 'heart_rate', 'systolic_bp', 'diastolic_bp']:
    if col in patient_vitals.columns:
        cols_to_save.append(col)
        break
# Note: When saving to CSV with index=False, you'll need to convert the index to a column first
# Example structure:
datetime_analysis = patient_vitals[['patient_id', 'year', 'month', 'day', 'days_since_start', 'temperature']].copy()
datetime_analysis.to_csv('output/q1_datetime_analysis.csv', index=False)

Visits on clinic business days: 365
Visits on weekends: 0
Total unique visit dates: 365


## Part 1.3: Time Zone Handling

**TODO: Handle time zones**

In [ ]:
import pandas as pd import pytz import os os.makedirs('output', exist_ok=True) # Current UTC time utc_time = pd.Timestamp.utcnow() print("Current UTC time:", utc_time) # Convert UTC to US Eastern eastern_time = utc_time.tz_localize('UTC').tz_convert('US/Eastern') print("Eastern Time:", eastern_time)
# Current UTC time (already timezone-aware)
utc_time = pd.Timestamp.now(tz='UTC')
print("Current UTC time:", utc_time)

# Convert to US Eastern
eastern_time = utc_time.tz_convert('US/Eastern')
print("Eastern Time:", eastern_time)


# Create timezone-aware DataFrame from patient_vitals
patient_vitals_tz = patient_vitals.copy()
patient_vitals_tz.index = patient_vitals_tz.index.tz_localize('UTC')

# Convert to Eastern time
patient_vitals_tz_eastern = patient_vitals_tz.copy()
patient_vitals_tz_eastern.index = patient_vitals_tz_eastern.index.tz_convert('US/Eastern')

# Handle daylight saving time transitions
dst_date_utc = pd.Timestamp('2023-03-12 10:00:00', tz='UTC')
dst_time_eastern = dst_date_utc.tz_convert('US/Eastern')
print("DST example - UTC:", dst_date_utc)
print("DST example - Eastern:", dst_time_eastern)

# Document timezone operations
timezone_report = f"""
Timezone Operations Report

1. Original timezone:
The original patient_vitals dataset contained naive datetime objects (no timezone information).

2. Localization method:
Naive datetime indices were localized to UTC using tz_localize('UTC') to make them timezone-aware. This ensures a consistent reference time across all sites.

3. Conversion:
The UTC timestamps were then converted to US/Eastern using tz_convert('US/Eastern') for local interpretation. This allows clinical staff in Eastern time zone to interpret visit times accurately.

4. DST handling:
Using UTC as the base timezone avoids ambiguity caused by daylight saving time transitions. For example, March 12, 2023, 10:00:00 UTC converts to 06:00:00 in US/Eastern before the DST jump and correctly handles the missing hour. Storing timestamps in UTC ensures calculations, comparisons, and merging across sites are unambiguous.

5. Example:
Original UTC timestamp: {dst_date_utc}
Converted Eastern timestamp: {dst_time_eastern}

Recommendation: Always store temporal data in UTC internally, and convert to local timezones only for reporting, visualization, or local operations. This prevents errors due to DST and maintains consistent longitudinal analysis across multiple sites.
"""

# Save report
with open('output/q1_timezone_report.txt', 'w') as f:
    f.write(timezone_report)

print("Timezone report saved to 'output/q1_timezone_report.txt'")


Current UTC time: 2025-11-12 19:58:03.620086+00:00


TypeError: Cannot localize tz-aware Timestamp, use tz_convert for conversions

## Submission Checklist

Before moving to Question 2, verify you've created:

- [ ] `output/q1_datetime_analysis.csv` - datetime analysis results
- [ ] `output/q1_timezone_report.txt` - timezone handling report
